# Experiment 1: Environment Configuration & NLP Library Verification

### Problem Statement:
Install and configure a Python-based NLP development environment using NLTK, SpaCy, Gensim, and Hugging Face Transformers. Verify the installation by executing sample NLP tasks and generating basic outputs.


In [1]:
import sys
import os
import platform
import torch
import nltk
import spacy
import gensim
import transformers
import pandas as pd
import numpy as np

# System & Environment Diagnostic
env_summary = {
    "Python Version": sys.version.split()[0],
    "Platform OS": platform.system(),
    "OS Release": platform.release(),
    "Architecture": platform.machine(),
    "PyTorch Version": torch.__version__,
    "MPS (Apple GPU) Available": torch.backends.mps.is_available(),
    "NLTK Version": nltk.__version__,
    "SpaCy Version": spacy.__version__,
    "Gensim Version": gensim.__version__,
    "Transformers Version": transformers.__version__
}

df_env = pd.DataFrame(list(env_summary.items()), columns=["Component / Metric", "Value / Status"])
print("=================== ENVIRONMENT CONFIGURATION REPORT ===================")
print(df_env.to_string(index=False))


/Users/jeet/miniconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=================== ENVIRONMENT CONFIGURATION REPORT ===================
       Component / Metric Value / Status
           Python Version        3.11.16
              Platform OS         Darwin
               OS Release         27.0.0
             Architecture          arm64
          PyTorch Version         2.14.1
MPS (Apple GPU) Available           True
             NLTK Version         3.10.3
            SpaCy Version         3.8.16
           Gensim Version          4.4.0
     Transformers Version         5.17.0


In [2]:
# Ensure NLTK corpora are downloaded
resources = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']
for res in resources:
    nltk.download(res, quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag

raw_text = (
    "Natural Language Processing empowers computers to understand human language. "
    "Dr. Emily is designing intelligent models at OpenAI in San Francisco."
)

# 1. Tokenization
sentences = sent_tokenize(raw_text)
words = word_tokenize(raw_text)
stop_words = set(stopwords.words('english'))
filtered_words = [w for w in words if w.lower() not in stop_words and w.isalnum()]

# 2. Part-of-Speech Tagging
pos_tags = pos_tag(filtered_words)

# 3. WordNet Exploration
synsets = wordnet.synsets("intelligence")
definitions = [(s.name(), s.definition(), s.lemma_names()[:3]) for s in synsets[:3]]

print(f"--- NLTK Tokenization & POS Tagging ---")
print(f"Total Sentences : {len(sentences)}")
print(f"Total Words     : {len(words)}")
print(f"Filtered Words  : {filtered_words}")
print(f"\nSample POS Tags (Penn Treebank):")
for token, tag in pos_tags[:8]:
    print(f"  {token:<15} -> {tag}")

print(f"\n--- WordNet Synset Lookup for 'intelligence' ---")
for name, definition, lemmas in definitions:
    print(f"  Synset: {name:<15} Def: {definition}")


--- NLTK Tokenization & POS Tagging ---
Total Sentences : 2
Total Words     : 22
Filtered Words  : ['Natural', 'Language', 'Processing', 'empowers', 'computers', 'understand', 'human', 'language', 'Emily', 'designing', 'intelligent', 'models', 'OpenAI', 'San', 'Francisco']

Sample POS Tags (Penn Treebank):
  Natural         -> JJ
  Language        -> NNP
  Processing      -> NNP
  empowers        -> VBZ
  computers       -> NNS
  understand      -> VBP
  human           -> JJ
  language        -> NN

--- WordNet Synset Lookup for 'intelligence' ---
  Synset: intelligence.n.01 Def: the ability to comprehend; to understand and profit from experience
  Synset: intelligence.n.02 Def: a unit responsible for gathering and interpreting information about an enemy
  Synset: intelligence.n.03 Def: secret information about an enemy (or potential enemy)


In [3]:
nlp = spacy.load("en_core_web_sm")

doc = nlp("Sundar Pichai announced that Google opened a new AI research lab in London on January 15, 2024.")

tokens_data = []
for token in doc:
    tokens_data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS": token.pos_,
        "Tag": token.tag_,
        "Dependency": token.dep_,
        "Head": token.head.text
    })

df_spacy_tokens = pd.DataFrame(tokens_data)
print("--- SpaCy Linguistic Token Annotations ---")
print(df_spacy_tokens.head(10).to_string(index=False))

print("\n--- SpaCy Named Entity Recognition ---")
entities_data = []
for ent in doc.ents:
    entities_data.append({
        "Entity": ent.text,
        "Label": ent.label_,
        "Explanation": spacy.explain(ent.label_)
    })

df_entities = pd.DataFrame(entities_data)
print(df_entities.to_string(index=False))


--- SpaCy Linguistic Token Annotations ---
    Token    Lemma   POS Tag Dependency      Head
   Sundar   Sundar PROPN NNP   compound    Pichai
   Pichai   Pichai PROPN NNP      nsubj announced
announced announce  VERB VBD       ROOT announced
     that     that SCONJ  IN       mark    opened
   Google   Google PROPN NNP      nsubj    opened
   opened     open  VERB VBD      ccomp announced
        a        a   DET  DT        det       lab
      new      new   ADJ  JJ       amod       lab
       AI       AI PROPN NNP   compound       lab
 research research  NOUN  NN   compound       lab

--- SpaCy Named Entity Recognition ---
          Entity Label                             Explanation
   Sundar Pichai   ORG Companies, agencies, institutions, etc.
          Google   ORG Companies, agencies, institutions, etc.
              AI   GPE               Countries, cities, states
          London   GPE               Countries, cities, states
January 15, 2024  DATE   Absolute or relative dates 

In [4]:
from gensim import corpora
from gensim.models import Word2Vec

mini_corpus = [
    ["deep", "learning", "neural", "networks", "revolutionize", "artificial", "intelligence"],
    ["natural", "language", "processing", "enables", "semantic", "text", "understanding"],
    ["transformers", "and", "large", "language", "models", "achieve", "superhuman", "accuracy"],
    ["artificial", "intelligence", "transforms", "healthcare", "and", "finance", "systems"],
    ["neural", "networks", "learn", "hierarchical", "feature", "representations", "from", "data"]
]

# 1. Dictionary & BoW Corpus
dictionary = corpora.Dictionary(mini_corpus)
bow_corpus = [dictionary.doc2bow(doc) for doc in mini_corpus]

print(f"--- Gensim Dictionary & Corpus ---")
print(f"Vocabulary Size : {len(dictionary)} unique tokens")
print(f"Sample BoW vector for Doc 0: {bow_corpus[0]}")

# 2. Train Word2Vec Model
w2v_model = Word2Vec(
    sentences=mini_corpus,
    vector_size=32,
    window=3,
    min_count=1,
    workers=2,
    epochs=100,
    seed=42
)

word1, word2 = "neural", "networks"
sim_score = w2v_model.wv.similarity(word1, word2)
vector_shape = w2v_model.wv["neural"].shape

print(f"\n--- Gensim Word2Vec Model ---")
print(f"Vector dimensionality : {vector_shape}")
print(f"Cosine similarity between '{word1}' and '{word2}': {sim_score:.4f}")


--- Gensim Dictionary & Corpus ---
Vocabulary Size : 31 unique tokens
Sample BoW vector for Doc 0: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1)]

--- Gensim Word2Vec Model ---
Vector dimensionality : (32,)
Cosine similarity between 'neural' and 'networks': 0.2544


In [5]:
from transformers import pipeline

sentiment_classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

test_sentences = [
    "The new NLP transformer architecture performs exceptionally well with rapid inference times!",
    "The training run encountered gradient explosions and completely failed to converge."
]

results = sentiment_classifier(test_sentences)

print("--- Hugging Face Transformers Inference ---")
for text, res in zip(test_sentences, results):
    print(f"Input Text : '{text}'")
    print(f"Sentiment  : {res['label']} | Confidence Score: {res['score']:.4f}\n")


Loading weights:   0%|                                 | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 104/104 [00:00<00:00, 7811.32it/s]

--- Hugging Face Transformers Inference ---
Input Text : 'The new NLP transformer architecture performs exceptionally well with rapid inference times!'
Sentiment  : POSITIVE | Confidence Score: 0.9998

Input Text : 'The training run encountered gradient explosions and completely failed to converge.'
Sentiment  : NEGATIVE | Confidence Score: 0.9998

